In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

26/02/26 15:37:54 WARN Utils: Your hostname, Rob resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/02/26 15:37:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/26 15:37:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/26 15:37:55 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/02/26 15:37:55 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/02/26 15:37:55 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [3]:
df_green = spark.read.parquet('../data/pq/green/*/*')

In [15]:
rdd = df_green \
    .select('lpep_pickup_datetime', 'PULocationID', 'total_amount') \
    .rdd

In [16]:
from datetime import datetime

In [17]:
start = datetime(year=2020, month=1, day=1)

def filter_outliers(row):
    return row.lpep_pickup_datetime >= start

In [18]:
rows = rdd.take(10)
row = rows[0]

In [19]:
row

Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 16, 23, 31, 44), PULocationID=65, total_amount=25.8)

In [32]:
def prepare_for_grouping(row): 
    hour = row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)
    zone = row.PULocationID
    key = (hour, zone)
    amount = row.total_amount
    count = 1
    value = (amount, count)
    return (key, value)

In [33]:
def calculate_revenue(left_value, right_value):
    left_amount, left_count = left_value
    right_amount, right_count = right_value
    output_amount = left_amount + right_amount
    output_count = left_count + right_count
    return (output_amount, output_count)

In [34]:
from collections import namedtuple

In [35]:
RevenueRow = namedtuple('RevenueRow', ['hour', 'zone', 'revenue', 'count'])

In [36]:
def unwrap(row):
    return RevenueRow(
        hour=row[0][0], 
        zone=row[0][1],
        revenue=row[1][0],
        count=row[1][1]
    )

In [42]:
from pyspark.sql import types

In [43]:
result_schema = types.StructType([
    types.StructField('hour', types.TimestampType(), True),
    types.StructField('zone', types.IntegerType(), True),
    types.StructField('revenue', types.DoubleType(), True),
    types.StructField('count', types.IntegerType(), True)
])

In [47]:
df_result = rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF(result_schema)

In [48]:
df_result.write.parquet('tmp/green-revenue')

26/02/26 16:25:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/26 16:25:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/02/26 16:25:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/02/26 16:25:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/02/26 16:25:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/02/26 16:25:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/02/26 16:25:47 WARN MemoryManager: Total allocation exceeds 95.

In [54]:
columns = ['VendorID', 'lpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'trip_distance']

duration_rdd = df_green \
    .select(columns) \
    .rdd

In [55]:
import pandas as pd

In [56]:
rows = duration_rdd.take(10)

In [60]:
pd.DataFrame(rows, columns=columns)

,VendorID,lpep_pickup_datetime,PULocationID,DOLocationID,trip_distance
0,2.0,2020-01-16 23:31:44,65,56,8.60
1,2.0,2020-01-14 15:01:42,129,141,5.16
2,NaN,2020-01-24 05:55:00,242,261,16.27
3,2.0,2020-01-31 08:19:03,166,238,1.42
4,NaN,2020-01-12 18:54:00,41,235,4.62
5,2.0,2020-01-11 17:13:41,66,49,2.67
6,2.0,2020-01-29 19:23:03,260,226,0.64
7,2.0,2020-01-26 11:45:13,116,116,0.43
8,2.0,2020-01-03 08:58:57,74,75,1.38
9,2.0,2020-01-27 21:56:19,41,74,1.00


In [57]:
df = pd.DataFrame(rows, columns=columns)

In [64]:
columns

['VendorID',
 'lpep_pickup_datetime',
 'PULocationID',
 'DOLocationID',
 'trip_distance']

In [65]:
#model = ...
def model_predict(df):
#     y_pred = model.predict(df)
    y_pred = df.trip_distance * 5
    return y_pred

In [66]:
def apply_model_in_batch(rows):
    df = pd.DataFrame(rows, columns=columns)
    predictions = model_predict(df)
    df['predicted_duration'] = predictions
    for row in df.itertuples():
        yield row

In [67]:
df_predicts = duration_rdd \
    .mapPartitions(apply_model_in_batch)\
    .toDF() \
    .drop('Index')

In [68]:
df_predicts.select('predicted_duration').show()

[Stage 27:>                                                         (0 + 1) / 1]

+------------------+
|predicted_duration|
+------------------+
|              43.0|
|              25.8|
|             81.35|
|               7.1|
|              23.1|
|             13.35|
|               3.2|
|              2.15|
|6.8999999999999995|
|               5.0|
|               7.1|
|41.150000000000006|
|             33.55|
|               0.0|
|             41.25|
|7.1499999999999995|
|              51.2|
|              3.05|
| 8.100000000000001|
|17.150000000000002|
+------------------+
only showing top 20 rows

